In [ ]:
import pandas as pd
import re
import nltk
import time
import numpy as np

# nltk.download('punkt_tab')
# nltk.download('stopwords', quiet=True)
# nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, LSTM, Input
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model

In [ ]:
data = pd.read_csv("arabic_english_dataset.csv")
data.head(3)

,english,arabic
0,hello,مرحبا
1,good morning,صباح الخير
2,good evening,مساء الخير


In [ ]:
data["arabic"] = ("<start> " + data["arabic"].astype(str) + " <end>")
data["arabic"].head()

,arabic
0,<start> مرحبا <end>
1,<start> صباح الخير <end>
2,<start> مساء الخير <end>
3,<start> كيف حالك <end>
4,<start> أنا بخير <end>


In [ ]:
arb_tokenizer = Tokenizer(filters="")
arb_tokenizer.fit_on_texts(data["arabic"])

arb_sequences = arb_tokenizer.texts_to_sequences(data["arabic"])

arb_vocab_size = len(arb_tokenizer.word_index) + 1

In [ ]:
eng_tokenizer = Tokenizer(filters="")
eng_tokenizer.fit_on_texts(data["english"])

eng_sequences = eng_tokenizer.texts_to_sequences(data["english"])

eng_vocab_size = len(eng_tokenizer.word_index) + 1

In [ ]:
max_eng_len = max(len(seq) for seq in eng_sequences)
max_arb_len = max(len(seq) for seq in arb_sequences)

print("English max length:", max_eng_len)
print("Arabic max length:", max_arb_len)

English max length: 4
Arabic max length: 6


In [ ]:
encoder_input = pad_sequences(
    eng_sequences,
    maxlen=max_eng_len,
    padding="post"
)

decoder_input = pad_sequences(
    [seq[:-1] for seq in arb_sequences],
    maxlen=max_arb_len - 1,
    padding="post"
)

decoder_target = pad_sequences(
    [seq[1:] for seq in arb_sequences],
    maxlen=max_arb_len - 1,
    padding="post"
)

decoder_target = np.expand_dims(decoder_target, -1)

In [ ]:
latent_dim = 256

encoder_inputs = Input(shape=(max_eng_len,))

encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=256,
    mask_zero=True
)(encoder_inputs)

_, state_h, state_c = LSTM(
    latent_dim,
    return_state=True,
    dropout=0.3,
    recurrent_dropout=0.3
)(encoder_embedding)

encoder_states = [state_h, state_c]

In [ ]:

decoder_inputs = Input(shape=(max_arb_len - 1,))

decoder_embedding = Embedding(
    input_dim=arb_vocab_size,
    output_dim=256,
    mask_zero=True
)(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    dropout=0.3,
    recurrent_dropout=0.3
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    arb_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 4, 256)    │     20,736 │ input_layer_12[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 4)         │          0 │ input_layer_12[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 5, 256)    │     20,224 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ [(None, 256),     │    525,312 │ embedding_12[0][… │
│                     │ (None, 256),      │            │ not_equal_4[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ [(None, 5, 256),  │    525,312 │ embedding_13[0][… │
│                     │ (None, 256),      │            │ lstm_12[0][1],    │
│                     │ (None, 256)]      │            │ lstm_12[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 5, 79)     │     20,303 │ lstm_13[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,111,887 (4.24 MB)

 Trainable params: 1,111,887 (4.24 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    validation_split=0.2,
    epochs=100,
    batch_size=64
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.0091 - loss: 4.3662 - val_accuracy: 0.2800 - val_loss: 4.3616
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.2909 - loss: 4.3485 - val_accuracy: 0.3200 - val_loss: 4.3514
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.3000 - loss: 4.3283 - val_accuracy: 0.3200 - val_loss: 4.3392
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.2909 - loss: 4.3065 - val_accuracy: 0.3200 - val_loss: 4.3241
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.2909 - loss: 4.2796 - val_accuracy: 0.3200 - val_loss: 4.3045
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.2909 - loss: 4.2428 - val_accuracy: 0.3200 - val_loss: 4.2787
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.2909 - loss: 4.2027 - val_accuracy: 0.3200 - val_loss: 4.2445
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.2909 - loss: 4.1435 - val_accuracy: 0.3200 - val_

In [ ]:
print("Training Accuracy:", history.history["accuracy"][-1]*100)
print("Validation Accuracy:", history.history["val_accuracy"][-1]*100)

Training Accuracy: 98.181813955307
Validation Accuracy: 35.999998450279236


In [ ]:
loss, accuracy = model.evaluate(
    [encoder_input, decoder_input],
    decoder_target,
    verbose=0
)

print("Accuracy:", accuracy*100)
print("Loss:", loss)

Accuracy: 88.14814686775208
Loss: 1.6478641033172607
